# Módulo 3: Agentes de IA
- **Sesión:** 02 — Frameworks de agentes: LangGraph y protocolo MCP
- **Fecha:** 28/10/2026



In [1]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os

In [2]:

# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv(override = True)

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)



## Datos

In [3]:
PEDIDOS = {
    "48213": "en ruta, llega el 26/10/2026 a Arequipa",
    "77100": "entregado el 10/10/2026 en Lima",
    "50555": "en preparación, llega el 30/10/2026 a Cusco",
}


## Ejercicio 1 — Un grafo con un solo nodo
### Concepto
El grafo más simple tiene un nodo entre `START` y `END`. Se construye en tres pasos:

1. **Estado:** una clase `TypedDict` declara los datos del grafo.
2. **Nodo:** una función que recibe el estado y devuelve un diccionario **solo con lo que cambia**.
3. **Grafo:** se agregan el nodo y las aristas, y se **compila** con `compile()`.

```
START ──▶ saludar ──▶ END
```

### Ejecutar

In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


# 1. Estado
class EstadoSaludo(TypedDict) :
    nombre : str
    saludo : str


# 2. Nodo 
def saludar(estado : EstadoSaludo ) -> dict:

    mensaje = f"""
          Hola {estado['nombre']} , Bienvenid@ a la TIENDA
        """
    return {"saludo": mensaje}

# 3. Grafo

grafo = StateGraph(EstadoSaludo)

# Agregar nodos
grafo.add_node("saludar",saludar)

# Agregar aristas
grafo.add_edge(START , "saludar")
grafo.add_edge("saludar", END)


app = grafo.compile()
res = app.invoke({"nombre": "Jaime"})

print(res)


{'nombre': 'Jaime', 'saludo': '\n          Hola Jaime , Bienvenid@ a la TIENDA\n        '}
